In [ ]:
import os
import numpy as np
import rasterio
from tqdm import tqdm
import gc

# ========= Paths =========
input_base = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Depth_Classification"
output_base = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Edge_Area_1km"
os.makedirs(output_base, exist_ok=True)

# ========= Parameters =========
aggregation_factor = 33  # ~1km blocks (33*30 ≈ 990m)

# ========= Block Sum Function =========
def block_sum_with_progress(data, block_size):
    h, w = data.shape
    new_h = h // block_size
    new_w = w // block_size
    result = np.zeros((new_h, new_w), dtype=np.int32)
    for i in tqdm(range(new_h), desc="Processing Rows"):
        for j in range(new_w):
            block = data[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
            result[i, j] = np.sum(block)
    return result

# ========= Loop over years =========
for year in range(1988, 2022):
    print(f"\n=== Processing year: {year} ===")
    
    input_path = os.path.join(input_base, f"LCMAP_CU_{year}_V13_LCPRI.tif")
    output_path = os.path.join(output_base, f"Exterior_Forest_Rate_{year}_1km.tif")

    if not os.path.exists(input_path):
        print(f"⚠️ Input file not found: {input_path}")
        continue

    # === Load forest depth classification ===
    with rasterio.open(input_path) as src:
        depth_data = src.read(1)
        depth_transform = src.transform
        depth_crs = src.crs

    # === New transform for 1 km blocks ===
    new_transform = rasterio.transform.Affine(
        depth_transform.a * aggregation_factor, depth_transform.b, depth_transform.c,
        depth_transform.d, depth_transform.e * aggregation_factor, depth_transform.f
    )

    # === Class masks ===
    valid_mask = np.isin(depth_data, [1, 2, 3, 4, 5])
    edge_mask = np.isin(depth_data, [1, 2, 3, 4]) & valid_mask
    total_mask = valid_mask  # forest = class 1–5

    # === Convert to counts ===
    edge_count = edge_mask.astype(np.uint16)
    total_count = total_mask.astype(np.uint16)

    del depth_data, edge_mask, total_mask, valid_mask
    gc.collect()

    # === Aggregate to 1km blocks ===
    agg_edge_count = block_sum_with_progress(edge_count, aggregation_factor)
    agg_total_count = block_sum_with_progress(total_count, aggregation_factor)

    del edge_count, total_count
    gc.collect()

    # === Exterior forest rate (ratio of edge to total) ===
    with np.errstate(divide='ignore', invalid='ignore'):
        exterior_rate = np.where(agg_total_count > 0,
                                  agg_edge_count / agg_total_count,
                                  np.nan)

    # === Save as float32 GeoTIFF ===
    meta = {
        "driver": "GTiff",
        "height": exterior_rate.shape[0],
        "width": exterior_rate.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": depth_crs,
        "transform": new_transform,
        "compress": "lzw"
    }

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(exterior_rate.astype(np.float32), 1)

    print(f"✅ Saved: {output_path}")